# Phase Functional: Per-Example Failure Analysis

## Overview
Characterize *how* circuits fail during cross-band transfer. Are failures bimodal
(some examples fully fail, others work fine) or uniform (all examples degrade slightly)?
This constrains the mechanistic explanation for transfer asymmetry.

## Key Questions
1. Are correct_prob distributions bimodal or uniform during cross-band transfer?
2. Are there "always-correct" or "always-wrong" examples across conditions?
3. Do LF->HF and HF->LF failures look qualitatively different?
4. Does example difficulty (base model confidence) predict transfer failure?

## Analysis Framework
- **Distribution characterization**: Hartigan's dip test for bimodality
- **Robustness scores**: Per-example fraction of circuits that get it right
- **Directional comparison**: LF->HF vs HF->LF failure modes
- **Difficulty analysis**: Failure rate vs base model confidence

## Notebook Structure
1. Setup & Loading
2. Distribution Characterization (bimodality)
3. Failure Pattern Analysis (robustness scores)
4. Directional Comparison
5. Visualizations
6. Statistical Tests
7. Export

## Data Sources
- Per-example JSONs from `lsc_per_example_eval.py` (300 files, ~225 examples each)
- `full_transfer_data.csv` from Notebook 01

## 1. Setup & Loading

In [1]:
import sys
import json
import warnings

warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

NOTEBOOK_DIR = Path("LSC_circuit_analysis/01_Phase_Functional")
sys.path.insert(0, str(NOTEBOOK_DIR))

from utils.constants import *
from utils.plotting import setup_plotting, save_figure

setup_plotting()
ANALYSIS_DIR, VIZ_DIR = get_output_dirs()

# Failure analysis output directories
FA_VIZ_DIR = VIZ_DIR / "failure_analysis"
FA_VIZ_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_SEED)
print(f"Analysis output: {ANALYSIS_DIR}")
print(f"Failure analysis viz: {FA_VIZ_DIR}")

Analysis output: LSC_circuit_analysis/01_Phase_Functional/outputs/analysis
Failure analysis viz: LSC_circuit_analysis/01_Phase_Functional/outputs/viz/failure_analysis


In [2]:
# Load per-example data from all 300 condition files
per_example_dir = PER_EXAMPLE_DIR
all_conditions = []
all_examples = []

for model in MODELS:
    m_safe = model.replace("-", "_")
    for band in BANDS:
        for draw in DRAWS:
            for test_band in BANDS:
                json_path = per_example_dir / m_safe / band / draw / f"{test_band}.json"
                if not json_path.exists():
                    continue
                with open(json_path) as f:
                    data = json.load(f)

                same_band = band == test_band
                examples = data.get("examples", [])
                probs = [e["correct_prob"] for e in examples]
                n_correct = sum(1 for e in examples if e["correct"])

                all_conditions.append(
                    {
                        "model": model,
                        "train_band": band,
                        "draw": draw,
                        "test_band": test_band,
                        "same_band": same_band,
                        "n_examples": len(examples),
                        "accuracy": n_correct / len(examples) if examples else 0,
                        "mean_correct_prob": np.mean(probs) if probs else 0,
                        "std_correct_prob": np.std(probs) if probs else 0,
                        "median_correct_prob": np.median(probs) if probs else 0,
                    }
                )

                for e in examples:
                    all_examples.append(
                        {
                            "model": model,
                            "train_band": band,
                            "draw": draw,
                            "test_band": test_band,
                            "same_band": same_band,
                            **e,
                        }
                    )

df_conditions = pd.DataFrame(all_conditions)
df_examples = pd.DataFrame(all_examples)
print(f"Conditions loaded: {len(df_conditions)}")
print(f"Total examples: {len(df_examples)}")
print(f"\nConditions per category:")
print(f"  Same-band: {df_conditions['same_band'].sum()}")
print(f"  Cross-band: {(~df_conditions['same_band']).sum()}")

Conditions loaded: 375
Total examples: 84375

Conditions per category:
  Same-band: 75
  Cross-band: 300


## 2. Distribution Characterization

Test whether correct_prob distributions are bimodal (some examples completely fail,
others work perfectly) or uniform (all examples degrade).

Using Hartigan's dip test for unimodality.

In [3]:
try:
    import diptest

    HAS_DIPTEST = True
except ImportError:
    HAS_DIPTEST = False
    print("diptest package not available. Install with: pip install diptest")
    print("Falling back to kurtosis-based bimodality coefficient.")


def bimodality_coefficient(data):
    """Sarle's bimodality coefficient: BC = (skew^2 + 1) / kurtosis.
    BC > 5/9 ~ 0.555 suggests bimodality."""
    n = len(data)
    if n < 4:
        return np.nan
    skew = stats.skew(data)
    kurt = stats.kurtosis(data, fisher=False)  # excess=False -> Pearson kurtosis
    if kurt == 0:
        return np.nan
    return (skew**2 + 1) / kurt


bimodality_results = []

for _, row in df_conditions.iterrows():
    mask = (
        (df_examples["model"] == row["model"])
        & (df_examples["train_band"] == row["train_band"])
        & (df_examples["draw"] == row["draw"])
        & (df_examples["test_band"] == row["test_band"])
    )
    probs = df_examples.loc[mask, "correct_prob"].values

    if len(probs) < 10:
        continue

    result = {
        "model": row["model"],
        "train_band": row["train_band"],
        "draw": row["draw"],
        "test_band": row["test_band"],
        "same_band": row["same_band"],
        "accuracy": row["accuracy"],
        "bc": bimodality_coefficient(probs),
    }

    if HAS_DIPTEST:
        dip_stat, dip_p = diptest.diptest(probs)
        result["dip_stat"] = dip_stat
        result["dip_pvalue"] = dip_p
        result["bimodal"] = dip_p < 0.05
    else:
        result["bimodal"] = result["bc"] > 0.555

    bimodality_results.append(result)

df_bimodality = pd.DataFrame(bimodality_results)
print(f"Bimodality analysis: {len(df_bimodality)} conditions")
print(f"\nBimodal conditions: {df_bimodality['bimodal'].sum()}/{len(df_bimodality)}")
print(
    f"  Same-band bimodal: {df_bimodality[df_bimodality['same_band']]['bimodal'].sum()}"
)
print(
    f"  Cross-band bimodal: {df_bimodality[~df_bimodality['same_band']]['bimodal'].sum()}"
)

diptest package not available. Install with: pip install diptest
Falling back to kurtosis-based bimodality coefficient.


Bimodality analysis: 375 conditions

Bimodal conditions: 310/375
  Same-band bimodal: 61
  Cross-band bimodal: 249


### F3_01: Correct Probability Distributions

In [4]:
# Compare same-band vs cross-band correct_prob distributions
n_models = len(MODELS)

ncols = 3

nrows = (n_models + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))
axes_flat = axes.flatten()

for idx, model in enumerate(MODELS):
    ax = axes_flat[idx]
    me = df_examples[df_examples["model"] == model]

    same = me[me["same_band"]]["correct_prob"].values
    cross = me[~me["same_band"]]["correct_prob"].values

    ax.hist(
        same,
        bins=50,
        alpha=0.6,
        color="steelblue",
        label=f"Same-band (n={len(same)})",
        density=True,
    )
    ax.hist(
        cross,
        bins=50,
        alpha=0.6,
        color="coral",
        label=f"Cross-band (n={len(cross)})",
        density=True,
    )
    ax.set_xlabel("P(correct)")
    ax.set_ylabel("Density")
    ax.set_title(f"{model}", fontsize=13)
    ax.legend(fontsize=9)

for idx in range(n_models, len(axes_flat)):
    axes_flat[idx].set_visible(False)


fig.suptitle(
    "Correct Probability Distributions: Same-Band vs Cross-Band", fontsize=15, y=1.02
)
fig.tight_layout()
save_figure(fig, "failure_analysis/F3_01_correct_prob_distributions.png")

  Saved: LSC_circuit_analysis/01_Phase_Functional/outputs/viz/failure_analysis/F3_01_correct_prob_distributions.png


### F3_02: Bimodality Heatmap

In [5]:
# Bimodality coefficient heatmap by model x (train_band -> test_band)
n_models = len(MODELS)

ncols = 3

nrows = (n_models + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))
axes_flat = axes.flatten()

metric_col = "dip_pvalue" if HAS_DIPTEST else "bc"
metric_label = "Dip test p-value" if HAS_DIPTEST else "Bimodality Coefficient"

for idx, model in enumerate(MODELS):
    ax = axes_flat[idx]
    mb = df_bimodality[df_bimodality["model"] == model]

    # Average over draws
    pivot = mb.groupby(["train_band", "test_band"])[metric_col].mean().reset_index()
    matrix = pivot.pivot(index="train_band", columns="test_band", values=metric_col)
    matrix = matrix.reindex(index=BANDS, columns=BANDS)

    band_labels = [BAND_NAMES[b] for b in BANDS]
    if HAS_DIPTEST:
        sns.heatmap(
            matrix,
            annot=True,
            fmt=".3f",
            cmap="RdYlGn",
            ax=ax,
            xticklabels=band_labels,
            yticklabels=band_labels,
            square=True,
            linewidths=0,
            linecolor="none",
            vmin=0,
            vmax=1,
        )
    else:
        sns.heatmap(
            matrix,
            annot=True,
            fmt=".3f",
            cmap="RdYlGn_r",
            ax=ax,
            xticklabels=band_labels,
            yticklabels=band_labels,
            square=True,
            linewidths=0,
            linecolor="none",
        )

    ax.set_title(f"{model}", fontsize=13)
    ax.set_xlabel("Test Band")
    ax.set_ylabel("Train Band")

for idx in range(n_models, len(axes_flat)):
    axes_flat[idx].set_visible(False)


fig.suptitle(f"{metric_label} by Transfer Condition", fontsize=15, y=1.02)
fig.tight_layout()
save_figure(fig, "failure_analysis/F3_02_bimodality_heatmap.png")

  Saved: LSC_circuit_analysis/01_Phase_Functional/outputs/viz/failure_analysis/F3_02_bimodality_heatmap.png


## 3. Failure Pattern Analysis

Compute per-example robustness scores: what fraction of circuits get each example right?

In [6]:
# Robustness: per (model, test_band, example_idx), how many circuits get it right?
robustness = (
    df_examples.groupby(["model", "test_band", "example_idx"])
    .agg(
        n_conditions=("correct", "count"),
        n_correct=("correct", "sum"),
        mean_prob=("correct_prob", "mean"),
    )
    .reset_index()
)

robustness["robustness_score"] = robustness["n_correct"] / robustness["n_conditions"]

# Classify examples
robustness["category"] = "variable"
robustness.loc[robustness["robustness_score"] == 1.0, "category"] = "always_correct"
robustness.loc[robustness["robustness_score"] == 0.0, "category"] = "always_wrong"

print("Robustness score distribution:")
for model in MODELS:
    mr = robustness[robustness["model"] == model]
    print(f"\n{model}:")
    cat_counts = mr["category"].value_counts()
    for cat in ["always_correct", "variable", "always_wrong"]:
        count = cat_counts.get(cat, 0)
        pct = count / len(mr) * 100
        print(f"  {cat}: {count} ({pct:.1f}%)")

Robustness score distribution:

pythia-70m:
  always_correct: 32 (2.8%)
  variable: 925 (82.2%)
  always_wrong: 168 (14.9%)

pythia-160m:
  always_correct: 579 (51.5%)
  variable: 546 (48.5%)
  always_wrong: 0 (0.0%)

pythia-410m:
  always_correct: 810 (72.0%)
  variable: 315 (28.0%)
  always_wrong: 0 (0.0%)

pythia-1b:
  always_correct: 561 (49.9%)
  variable: 564 (50.1%)
  always_wrong: 0 (0.0%)

pythia-1.4b:
  always_correct: 421 (37.4%)
  variable: 704 (62.6%)
  always_wrong: 0 (0.0%)


### F3_03: Example Robustness Distribution

In [7]:
n_models = len(MODELS)
ncols = 3
nrows = (n_models + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))
axes_flat = axes.flatten()

for idx, model in enumerate(MODELS):
    ax = axes_flat[idx]
    mr = robustness[robustness["model"] == model]

    ax.hist(
        mr["robustness_score"], bins=20, edgecolor="black", alpha=0.7, color="steelblue"
    )
    ax.axvline(1.0, color="green", linestyle="--", alpha=0.7, label="Always correct")
    ax.axvline(0.0, color="red", linestyle="--", alpha=0.7, label="Always wrong")
    ax.set_xlabel("Robustness Score (fraction of circuits correct)")
    ax.set_ylabel("Count")
    ax.set_title(f"{model}", fontsize=13)
    ax.legend(fontsize=9)

for idx in range(n_models, len(axes_flat)):
    axes_flat[idx].set_visible(False)


fig.suptitle("Per-Example Robustness Score Distribution", fontsize=15, y=1.02)
fig.tight_layout()
save_figure(fig, "failure_analysis/F3_03_example_robustness.png")

  Saved: LSC_circuit_analysis/01_Phase_Functional/outputs/viz/failure_analysis/F3_03_example_robustness.png


## 4. Directional Comparison

Do LF->HF and HF->LF transfers produce qualitatively different failure patterns?

In [8]:
# Compare failure distributions for LF->HF vs HF->LF
lf_to_hf_mask = df_examples["train_band"].isin(LOW_FREQ_BANDS) & df_examples[
    "test_band"
].isin(HIGH_FREQ_BANDS)
hf_to_lf_mask = df_examples["train_band"].isin(HIGH_FREQ_BANDS) & df_examples[
    "test_band"
].isin(LOW_FREQ_BANDS)

for model in MODELS:
    me = df_examples[df_examples["model"] == model]
    lf_hf_probs = me[lf_to_hf_mask & (df_examples["model"] == model)][
        "correct_prob"
    ].values
    hf_lf_probs = me[hf_to_lf_mask & (df_examples["model"] == model)][
        "correct_prob"
    ].values

    if len(lf_hf_probs) > 0 and len(hf_lf_probs) > 0:
        ks_stat, ks_p = stats.ks_2samp(lf_hf_probs, hf_lf_probs)
        print(
            f"{model}: LF->HF mean_prob={np.mean(lf_hf_probs):.4f} vs "
            f"HF->LF mean_prob={np.mean(hf_lf_probs):.4f}, "
            f"KS={ks_stat:.4f}, p={ks_p:.6f}"
        )

pythia-70m: LF->HF mean_prob=0.1553 vs HF->LF mean_prob=0.1025, KS=0.2011, p=0.000000
pythia-160m: LF->HF mean_prob=0.4696 vs HF->LF mean_prob=0.2977, KS=0.2774, p=0.000000
pythia-410m: LF->HF mean_prob=0.4772 vs HF->LF mean_prob=0.3293, KS=0.2526, p=0.000000
pythia-1b: LF->HF mean_prob=0.5418 vs HF->LF mean_prob=0.3275, KS=0.2915, p=0.000000
pythia-1.4b: LF->HF mean_prob=0.4435 vs HF->LF mean_prob=0.1957, KS=0.3907, p=0.000000


### F3_04: Failure by Difficulty

In [9]:
# Bin examples by correct_prob on same-band (proxy for difficulty)
# Then check cross-band failure rate in each difficulty bin
n_models = len(MODELS)

ncols = 3

nrows = (n_models + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))
axes_flat = axes.flatten()

for idx, model in enumerate(MODELS):
    ax = axes_flat[idx]
    me = df_examples[df_examples["model"] == model]

    # Same-band correct_prob as difficulty proxy
    same_probs = me[me["same_band"]].groupby("example_idx")["correct_prob"].mean()
    cross_correct = me[~me["same_band"]].groupby("example_idx")["correct"].mean()

    # Merge on example_idx
    merged = pd.DataFrame(
        {
            "same_band_prob": same_probs,
            "cross_band_accuracy": cross_correct,
        }
    ).dropna()

    if len(merged) > 10:
        # Bin by difficulty
        merged["difficulty_bin"] = pd.qcut(
            merged["same_band_prob"],
            q=5,
            labels=["Very Hard", "Hard", "Medium", "Easy", "Very Easy"],
        )
        bin_means = merged.groupby("difficulty_bin", observed=True)[
            "cross_band_accuracy"
        ].mean()

        ax.bar(range(len(bin_means)), bin_means.values, color="steelblue", alpha=0.8)
        ax.set_xticks(range(len(bin_means)))
        ax.set_xticklabels(bin_means.index, rotation=30, ha="right")
        ax.set_ylabel("Cross-Band Accuracy")
        ax.set_title(f"{model}", fontsize=13)

        # Spearman
        rho, p = stats.spearmanr(
            merged["same_band_prob"], merged["cross_band_accuracy"]
        )
        ax.text(
            0.95,
            0.05,
            f"rho={rho:.3f}\np={p:.4f}",
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=9,
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
        )
    else:
        ax.set_title(f"{model} (insufficient data)")

for idx in range(n_models, len(axes_flat)):
    axes_flat[idx].set_visible(False)


fig.suptitle("Cross-Band Accuracy by Same-Band Difficulty", fontsize=15, y=1.02)
fig.tight_layout()
save_figure(fig, "failure_analysis/F3_04_failure_by_difficulty.png")

  Saved: LSC_circuit_analysis/01_Phase_Functional/outputs/viz/failure_analysis/F3_04_failure_by_difficulty.png


## 5. Statistical Tests

Standalone tests (not included in NB02 FDR correction: these are exploratory).

In [10]:
# KS test: same-band vs cross-band correct_prob distributions
print("KS Test: Same-Band vs Cross-Band correct_prob distributions")
print("=" * 70)
for model in MODELS:
    me = df_examples[df_examples["model"] == model]
    same = me[me["same_band"]]["correct_prob"].values
    cross = me[~me["same_band"]]["correct_prob"].values
    ks_stat, ks_p = stats.ks_2samp(same, cross)
    print(f"{model}: KS={ks_stat:.4f}, p={ks_p:.2e}")

# Spearman: example difficulty vs transfer failure rate
print(f"\nSpearman: Same-Band P(correct) vs Cross-Band Accuracy")
print("=" * 70)
for model in MODELS:
    me = df_examples[df_examples["model"] == model]
    same_probs = me[me["same_band"]].groupby("example_idx")["correct_prob"].mean()
    cross_correct = me[~me["same_band"]].groupby("example_idx")["correct"].mean()
    merged = pd.DataFrame({"same": same_probs, "cross": cross_correct}).dropna()
    if len(merged) > 10:
        rho, p = stats.spearmanr(merged["same"], merged["cross"])
        print(f"{model}: rho={rho:.4f}, p={p:.2e}")

KS Test: Same-Band vs Cross-Band correct_prob distributions
pythia-70m: KS=0.0263, p=4.69e-02
pythia-160m: KS=0.0508, p=1.69e-06
pythia-410m: KS=0.0859, p=9.20e-18
pythia-1b: KS=0.0927, p=1.26e-20
pythia-1.4b: KS=0.0820, p=3.06e-16

Spearman: Same-Band P(correct) vs Cross-Band Accuracy
pythia-70m: rho=0.7662, p=1.05e-44
pythia-160m: rho=0.5397, p=2.08e-18
pythia-410m: rho=0.3739, p=7.10e-09
pythia-1b: rho=0.3690, p=1.15e-08
pythia-1.4b: rho=0.4751, p=4.54e-14


## 6. Export

In [11]:
# Save analysis results
df_bimodality.to_csv(ANALYSIS_DIR / "bimodality_results.csv", index=False)
print(f"Saved: bimodality_results.csv ({len(df_bimodality)} rows)")

robustness.to_csv(ANALYSIS_DIR / "per_example_robustness.csv", index=False)
print(f"Saved: per_example_robustness.csv ({len(robustness)} rows)")

# Summary
failure_summary = df_conditions.copy()
failure_summary.to_csv(ANALYSIS_DIR / "failure_analysis_summary.csv", index=False)
print(f"Saved: failure_analysis_summary.csv ({len(failure_summary)} rows)")

print(f"\nVisualization files:")
for f in sorted(FA_VIZ_DIR.glob("*.png")):
    print(f"  {f.name}")

Saved: bimodality_results.csv (375 rows)
Saved: per_example_robustness.csv (5625 rows)
Saved: failure_analysis_summary.csv (375 rows)

Visualization files:
  F3_01_correct_prob_distributions.png
  F3_02_bimodality_heatmap.png
  F3_03_example_robustness.png
  F3_04_failure_by_difficulty.png


## Summary

In [12]:
print("=" * 80)
print("FAILURE ANALYSIS SUMMARY")
print("=" * 80)

print(f"\nConditions analyzed: {len(df_conditions)}")
print(f"Total per-example observations: {len(df_examples)}")

print(f"\nBimodality:")
print(f"  Bimodal conditions: {df_bimodality['bimodal'].sum()}/{len(df_bimodality)}")
n_same_bimodal = df_bimodality[df_bimodality["same_band"]]["bimodal"].sum()
n_same_total = df_bimodality["same_band"].sum()
n_cross_bimodal = df_bimodality[~df_bimodality["same_band"]]["bimodal"].sum()
n_cross_total = (~df_bimodality["same_band"]).sum()
print(f"  Same-band: {n_same_bimodal}/{n_same_total}")
print(f"  Cross-band: {n_cross_bimodal}/{n_cross_total}")

print(f"\nExample robustness categories (per model):")
for model in MODELS:
    mr = robustness[robustness["model"] == model]
    always_c = (mr["category"] == "always_correct").sum()
    always_w = (mr["category"] == "always_wrong").sum()
    variable = (mr["category"] == "variable").sum()
    print(
        f"  {model}: always_correct={always_c}, variable={variable}, always_wrong={always_w}"
    )

FAILURE ANALYSIS SUMMARY

Conditions analyzed: 375
Total per-example observations: 84375

Bimodality:
  Bimodal conditions: 310/375
  Same-band: 61/75
  Cross-band: 249/300

Example robustness categories (per model):
  pythia-70m: always_correct=32, variable=925, always_wrong=168
  pythia-160m: always_correct=579, variable=546, always_wrong=0
  pythia-410m: always_correct=810, variable=315, always_wrong=0
  pythia-1b: always_correct=561, variable=564, always_wrong=0
  pythia-1.4b: always_correct=421, variable=704, always_wrong=0
